# 12. Mask2Former - river plume segmentation

Fine-tuning of `facebook/mask2former-swin-small-ade-semantic` (HuggingFace
Transformers) for binary river-plume segmentation.

Setup used in the study: input 640x640, AdamW (lr 5e-5, weight decay 0.01),
CosineAnnealingWarmRestarts (T_0 = 10, T_mult = 2), gradient clipping 1.0,
batch size 2, up to 100 epochs with early stopping (patience 15 on validation
Dice). The Swin backbone is frozen; training uses the shared augmentation
policy of the study (flips, rotation within 25 degrees, adaptive photometric
transforms - identical to the CNN models).

The semantic plume-probability map is aggregated over all queries, weighting
each query's mask by its class probability (`aggregate_plume_probability`);
the same aggregation defines the training metric, the validation metric
(which drives early stopping and supplies the ensemble weight) and the
exported test predictions.


In [ ]:
import gc
import os
import random
import time
from pathlib import Path
from typing import Dict, List, Optional, Tuple

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from tqdm import tqdm
from transformers import Mask2FormerForUniversalSegmentation, Mask2FormerImageProcessor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

## 1. Configuration

In [ ]:
# ============================================================================
# CONFIGURATION - values reproduce the training run used in the paper
# ============================================================================

class Config:
    PRETRAINED = "facebook/mask2former-swin-small-ade-semantic"

    TRAIN_DIR = "split_data/train"
    VAL_DIR = "split_data/val"
    TEST_IMG_DIR = "split_data/test/images"
    TEST_MASK_DIR = "split_data/test/Masks"

    EPOCHS = 100
    BATCH_SIZE = 2
    IMG_SIZE = 640
    NUM_WORKERS = 0

    BASE_LR = 5e-5
    WEIGHT_DECAY = 0.01
    GRAD_CLIP = 1.0
    FREEZE_BACKBONE = True          # the Swin backbone stays frozen for the whole run

    PATIENCE = 15                   # early stopping on validation Dice

    PROJECT_DIR = "mask2former_results"
    SAVE_PATH = f"{PROJECT_DIR}/best_mask2former_model.pth"
    SAVE_HISTORY = True
    SAVE_PLOTS = True
    SAVE_CHECKPOINTS = True

    MAX_SAMPLES = None              # optional cap for quick debugging
    VERBOSE = True

    PRED_THRESHOLD = 0.5
    PROBABILITY_EXPORT_DIR = "probability_masks/test/mask2former"
    BINARY_EXPORT_DIR = "test_masks/mask2former"


os.makedirs(Config.PROJECT_DIR, exist_ok=True)


class Utils:
    @staticmethod
    def get_device() -> torch.device:
        device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
        if Config.VERBOSE:
            print(f"Using device: {device}")
            if torch.cuda.is_available():
                print(f"GPU: {torch.cuda.get_device_name()}")
        return device

    @staticmethod
    def clear_memory():
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        gc.collect()

    @staticmethod
    def save_plots(history: Dict, save_path: str):
        fig, ((ax1, ax2), (ax3, ax4)) = plt.subplots(2, 2, figsize=(15, 10))
        epochs = range(1, len(history["train_loss"]) + 1)

        ax1.plot(epochs, history["train_loss"], "b-", label="Train Loss")
        ax1.plot(epochs, history["val_loss"], "r-", label="Val Loss")
        ax1.set_title("Loss")
        ax1.set_xlabel("Epochs")
        ax1.legend()

        ax2.plot(epochs, history["train_dice"], "b-", label="Train Dice")
        ax2.plot(epochs, history["val_dice"], "r-", label="Val Dice")
        ax2.set_title("Dice")
        ax2.set_xlabel("Epochs")
        ax2.legend()

        ax3.plot(epochs, history["learning_rates"], "g-")
        ax3.set_title("Learning rate")
        ax3.set_xlabel("Epochs")
        ax3.set_yscale("log")

        ax4.axis("off")

        plt.tight_layout()
        plt.savefig(save_path, dpi=150)
        plt.show()

In [ ]:
# ============================================================================
# SEMANTIC AGGREGATION OF MASK2FORMER OUTPUTS
# ============================================================================
# Mask2Former predicts Q query masks plus a class distribution per query
# (num_labels object classes + one "no object" bin). Query 0 is NOT the plume
# mask by construction: the query that carries the plume is selected
# dynamically by the Hungarian matching and can differ between images and
# checkpoints. The semantic probability map must therefore aggregate ALL
# queries, weighting each query mask by the probability that the query
# belongs to the plume class - the same class-times-mask product used by the
# official Mask2FormerImageProcessor.post_process_semantic_segmentation.
#
# With a single foreground class in the ground truth (class_labels = [1]),
# the background class 0 never receives training targets, so the background
# channel of the official 2-channel semantic map is uninformative; the plume
# channel is used directly and clamped to [0, 1].

PLUME_CLASS_ID = 1  # must match class_labels in Mask2FormerDataset


def aggregate_plume_probability(outputs, plume_class_id=PLUME_CLASS_ID):
    """Plume probability map aggregated over all queries.

        p(pixel) = clamp( sum_q softmax(class_logits_q)[plume] * sigmoid(mask_q) )

    Returns a float tensor of shape [B, h, w] in [0, 1].
    """
    mask_probs = outputs.masks_queries_logits.sigmoid()                    # [B, Q, h, w]
    class_probs = outputs.class_queries_logits.softmax(dim=-1)[..., :-1]   # [B, Q, C], drop "no object"
    plume_weight = class_probs[..., plume_class_id]                        # [B, Q]
    prob = torch.einsum("bq,bqhw->bhw", plume_weight, mask_probs)
    return prob.clamp(0.0, 1.0)


## 2. Augmentation (shared study-wide policy)

In [ ]:
class Augmentor:
    """Shared augmentation policy of the study.

    Identical for every model except YOLO11-seg (which uses the native
    Ultralytics augmentation). Operates on RGB images in [0, 255] (uint8 or
    float32) and binary masks; returns an RGB float32 image in [0, 255] and a
    2D uint8 mask in {0, 1}. Photometric parameter ranges depend on the mean
    brightness of the image so that dark scenes are not darkened further and
    bright scenes are not over-exposed.
    """

    def __init__(self, rotation_range=(-25, 25), rotation_prob=0.9, flip_prob=0.5,
                 min_mean_allowed=60.0):
        self.rotation_range = rotation_range
        self.rotation_prob = rotation_prob
        self.flip_prob = flip_prob
        self.min_mean_allowed = min_mean_allowed

    def augment(self, img_rgb, mask):
        img = np.clip(img_rgb, 0, 255).astype(np.uint8)
        mask_in = (mask > 0.5).astype(np.uint8)

        h, w = img.shape[:2]

        # 1) Geometric augmentations.
        if np.random.rand() < self.flip_prob:
            img = cv2.flip(img, 1)
            mask_in = cv2.flip(mask_in, 1)
        if np.random.rand() < self.flip_prob:
            img = cv2.flip(img, 0)
            mask_in = cv2.flip(mask_in, 0)

        if np.random.rand() < self.rotation_prob:
            angle = float(np.random.uniform(self.rotation_range[0], self.rotation_range[1]))
            Mrot = cv2.getRotationMatrix2D((w / 2.0, h / 2.0), angle, 1.0)
            img = cv2.warpAffine(img, Mrot, (w, h), flags=cv2.INTER_LINEAR,
                                 borderMode=cv2.BORDER_CONSTANT, borderValue=(0, 0, 0))
            mask_in = cv2.warpAffine(mask_in, Mrot, (w, h), flags=cv2.INTER_NEAREST,
                                     borderMode=cv2.BORDER_CONSTANT, borderValue=0)

        # 2) Adaptive photometric transforms driven by the mean brightness.
        gray0 = cv2.cvtColor(img, cv2.COLOR_RGB2GRAY).astype(np.float32)
        mean0 = float(gray0.mean())

        dark_thresh = 80.0
        bright_thresh = 200.0

        if mean0 < dark_thresh:
            # Dark scenes: brighten gently, never darken.
            gamma = float(np.random.uniform(0.8, 1.05))
            alpha = float(np.random.uniform(0.9, 1.15))
            beta = float(np.random.uniform(0, 40))
            apply_clahe = (np.random.rand() < 0.3)
            apply_hsv = (np.random.rand() < 0.25)
        elif mean0 > bright_thresh:
            # Very bright scenes: allow slight darkening.
            gamma = float(np.random.uniform(0.95, 1.25))
            alpha = float(np.random.uniform(0.85, 1.25))
            beta = float(np.random.uniform(-40, 20))
            apply_clahe = (np.random.rand() < 0.5)
            apply_hsv = (np.random.rand() < 0.5)
        else:
            # Mid-tone scenes: moderate transforms.
            gamma = float(np.random.uniform(0.9, 1.1))
            alpha = float(np.random.uniform(0.92, 1.2))
            beta = float(np.random.uniform(-20, 30))
            apply_clahe = (np.random.rand() < 0.45)
            apply_hsv = (np.random.rand() < 0.4)

        # Gamma correction in [0, 1] float space.
        img_f = np.clip(img.astype(np.float32) / 255.0, 0.0, 1.0)
        if abs(gamma - 1.0) > 1e-6:
            img_f = np.power(img_f, gamma)
        img_gc = np.clip(img_f * 255.0, 0, 255).astype(np.float32)

        # Mean-centred linear contrast plus brightness shift.
        gray = cv2.cvtColor(img_gc.astype(np.uint8), cv2.COLOR_RGB2GRAY).astype(np.float32)
        mean_l = gray.mean()
        img_out = (img_gc - mean_l) * alpha + mean_l + beta

        # Moderate CLAHE on the L channel.
        if apply_clahe:
            lab = cv2.cvtColor(np.clip(img_out, 0, 255).astype(np.uint8), cv2.COLOR_RGB2LAB)
            L, A, B = cv2.split(lab)
            clip_limit = np.random.uniform(1.0, 3.0)
            clahe = cv2.createCLAHE(clipLimit=clip_limit, tileGridSize=(8, 8))
            L = clahe.apply(L)
            lab = cv2.merge([L, A, B])
            img_out = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB).astype(np.float32)

        # Gentle HSV tweak.
        if apply_hsv:
            hsv = cv2.cvtColor(np.clip(img_out, 0, 255).astype(np.uint8),
                               cv2.COLOR_RGB2HSV).astype(np.float32)
            h_ch, s_ch, v_ch = cv2.split(hsv)
            if mean0 < dark_thresh:
                sat_mul = np.random.uniform(0.9, 1.15)
                val_shift = np.random.uniform(0, 30)
            else:
                sat_mul = np.random.uniform(0.85, 1.25)
                val_shift = np.random.uniform(-30, 30)
            s_ch = np.clip(s_ch * sat_mul, 0, 255)
            v_ch = np.clip(v_ch + val_shift, 0, 255)
            hsv2 = cv2.merge([h_ch, s_ch, v_ch]).astype(np.uint8)
            img_out = cv2.cvtColor(hsv2, cv2.COLOR_HSV2RGB).astype(np.float32)

        # Strict guard against overly dark outputs.
        gray_after = cv2.cvtColor(np.clip(img_out, 0, 255).astype(np.uint8), cv2.COLOR_RGB2GRAY)
        mean_after = float(gray_after.mean())
        if mean_after < self.min_mean_allowed:
            img_out = img_out + (self.min_mean_allowed - mean_after)

        img_final = np.clip(img_out, 0, 255).astype(np.float32)
        mask_final = (mask_in > 0).astype(np.uint8)

        return img_final, mask_final

## 3. Dataset and collate function

In [ ]:
class Mask2FormerDataset(Dataset):
    """Image/mask dataset producing Mask2Former training inputs.

    Masks are binarized at 127, resampled to 1/4 of the input resolution (the
    scale expected by the mask head) and paired with a single class label 1.
    Training samples are augmented with the shared policy of the study.
    """

    def __init__(self, images_dir: Path, masks_dir: Path, processor: Mask2FormerImageProcessor,
                 img_size: int = Config.IMG_SIZE, augment: bool = False,
                 max_samples: Optional[int] = None):
        self.images_dir = Path(images_dir)
        self.masks_dir = Path(masks_dir)
        self.processor = processor
        self.img_size = img_size
        self.augment = augment

        # Training samples are augmented with the shared study-wide policy;
        # validation samples are only resized.
        self.augmentor = Augmentor() if augment else None

        self.image_mask_pairs = self._find_image_mask_pairs()

        if max_samples:
            self.image_mask_pairs = self.image_mask_pairs[:max_samples]

        print(f"Dataset: {len(self.image_mask_pairs)} image-mask pairs")

    def _find_image_mask_pairs(self) -> List[Tuple[Path, Path]]:
        image_extensions = {".png", ".jpg", ".jpeg", ".tif", ".tiff"}
        image_files = sorted([
            p for p in self.images_dir.glob("*")
            if p.suffix.lower() in image_extensions
        ])

        pairs = []
        for img_path in image_files:
            for ext in image_extensions:
                mask_path = self.masks_dir / (img_path.stem + ext)
                if mask_path.exists():
                    pairs.append((img_path, mask_path))
                    break
        return pairs

    def __len__(self) -> int:
        return len(self.image_mask_pairs)

    def __getitem__(self, idx: int) -> Dict[str, torch.Tensor]:
        img_path, mask_path = self.image_mask_pairs[idx]

        try:
            image = cv2.imread(str(img_path))
            image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
            mask = (mask > 127).astype(np.uint8)

            # Shared augmentation policy (training split only).
            if self.augmentor is not None:
                image, mask = self.augmentor.augment(image.astype(np.float32), mask)

            # Resize to the working resolution.
            image = cv2.resize(image, (self.img_size, self.img_size),
                               interpolation=cv2.INTER_LINEAR)
            mask = cv2.resize(mask, (self.img_size, self.img_size),
                              interpolation=cv2.INTER_NEAREST)
            image = np.clip(image, 0, 255).astype(np.uint8)

            # The processor handles normalization only; masks are prepared
            # manually below.
            inputs = self.processor(images=Image.fromarray(image), return_tensors="pt")

            pixel_values = inputs["pixel_values"].squeeze(0)
            pixel_mask = inputs["pixel_mask"].squeeze(0)

            mask_labels = torch.from_numpy(mask).unsqueeze(0).float()  # [1, H, W]

            # Resample to the resolution expected by the mask head (H/4).
            target_size = pixel_values.shape[-1] // 4
            mask_labels = F.interpolate(
                mask_labels.unsqueeze(0),
                size=(target_size, target_size),
                mode="nearest",
            ).squeeze(0)

            class_labels = torch.tensor([1], dtype=torch.long)  # single object class

            return {
                "pixel_values": pixel_values,
                "pixel_mask": pixel_mask,
                "mask_labels": mask_labels,
                "class_labels": class_labels,
            }

        except Exception as e:
            print(f"Error loading {img_path}: {e}")
            dummy_size = Config.IMG_SIZE
            return {
                "pixel_values": torch.zeros((3, dummy_size, dummy_size)),
                "pixel_mask": torch.ones((dummy_size, dummy_size)),
                "mask_labels": torch.zeros((1, dummy_size // 4, dummy_size // 4)),
                "class_labels": torch.tensor([0], dtype=torch.long),
            }


def collate_fn(batch):
    """Stack tensors, padding mask/class labels to the largest count."""
    pixel_values = torch.stack([item["pixel_values"] for item in batch])
    pixel_mask = torch.stack([item["pixel_mask"] for item in batch])

    max_masks = max(item["mask_labels"].shape[0] for item in batch)

    mask_labels_list = []
    for item in batch:
        mask = item["mask_labels"]
        if mask.shape[0] < max_masks:
            padding = torch.zeros(max_masks - mask.shape[0], mask.shape[1], mask.shape[2],
                                  dtype=mask.dtype)
            mask = torch.cat([mask, padding], dim=0)
        mask_labels_list.append(mask)
    mask_labels = torch.stack(mask_labels_list)

    class_labels_list = []
    for item in batch:
        labels = item["class_labels"]
        if labels.shape[0] < max_masks:
            padding = torch.zeros(max_masks - labels.shape[0], dtype=labels.dtype)
            labels = torch.cat([labels, padding], dim=0)
        class_labels_list.append(labels)
    class_labels = torch.stack(class_labels_list)

    return {
        "pixel_values": pixel_values,
        "pixel_mask": pixel_mask,
        "mask_labels": mask_labels,
        "class_labels": class_labels,
    }


class Metrics:
    @staticmethod
    def dice_coefficient(pred: torch.Tensor, target: torch.Tensor,
                         threshold: float = 0.5, smooth: float = 1e-6) -> float:
        pred_binary = (pred > threshold).float()
        target_binary = (target > threshold).float()

        intersection = (pred_binary * target_binary).sum()
        union = pred_binary.sum() + target_binary.sum()

        dice = (2.0 * intersection + smooth) / (union + smooth)
        return dice.item()

## 4. Trainer

In [ ]:
class Mask2FormerTrainer:
    def __init__(self):
        self.device = Utils.get_device()
        self.model = None
        self.processor = None
        self.optimizer = None
        self.scheduler = None
        self.train_loader = None
        self.val_loader = None
        self.history = {
            "train_loss": [], "val_loss": [],
            "train_dice": [], "val_dice": [],
            "learning_rates": [],
        }

    def setup_data(self):
        print("\n" + "=" * 60)
        print("DATA SETUP")
        print("=" * 60)

        train_images = Path(Config.TRAIN_DIR) / "images"
        train_masks = Path(Config.TRAIN_DIR) / "Masks"
        val_images = Path(Config.VAL_DIR) / "images"
        val_masks = Path(Config.VAL_DIR) / "Masks"

        for path in [train_images, train_masks, val_images, val_masks]:
            if not path.exists():
                raise FileNotFoundError(f"Not found: {path}")

        self.processor = Mask2FormerImageProcessor.from_pretrained(
            Config.PRETRAINED,
            do_resize=False,
            do_normalize=True,
        )

        train_dataset = Mask2FormerDataset(
            train_images, train_masks, self.processor,
            augment=True, max_samples=Config.MAX_SAMPLES,
        )
        val_dataset = Mask2FormerDataset(
            val_images, val_masks, self.processor,
            augment=False,
            max_samples=Config.MAX_SAMPLES // 4 if Config.MAX_SAMPLES else None,
        )

        self.train_loader = DataLoader(
            train_dataset, batch_size=Config.BATCH_SIZE, shuffle=True,
            num_workers=Config.NUM_WORKERS, pin_memory=True,
            collate_fn=collate_fn, drop_last=True,
        )
        self.val_loader = DataLoader(
            val_dataset, batch_size=Config.BATCH_SIZE, shuffle=False,
            num_workers=Config.NUM_WORKERS, pin_memory=True,
            collate_fn=collate_fn,
        )

        print(f"Training: {len(train_dataset)} samples")
        print(f"Validation: {len(val_dataset)} samples")

    def setup_model(self):
        print("\n" + "=" * 60)
        print("MODEL SETUP")
        print("=" * 60)

        self.model = Mask2FormerForUniversalSegmentation.from_pretrained(
            Config.PRETRAINED,
            num_labels=2,
            ignore_mismatched_sizes=True,
        ).to(self.device)

        if Config.FREEZE_BACKBONE:
            frozen_params = 0
            for name, param in self.model.named_parameters():
                if "model.pixel_level_module.encoder" in name:
                    param.requires_grad = False
                    frozen_params += param.numel()
            print(f"Backbone frozen: {frozen_params:,} parameters")

        total_params = sum(p.numel() for p in self.model.parameters())
        trainable_params = sum(p.numel() for p in self.model.parameters() if p.requires_grad)
        print(f"Total: {total_params:,} | Trainable: {trainable_params:,}")

    def setup_optimizer(self):
        self.optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, self.model.parameters()),
            lr=Config.BASE_LR,
            weight_decay=Config.WEIGHT_DECAY,
        )

        self.scheduler = torch.optim.lr_scheduler.CosineAnnealingWarmRestarts(
            self.optimizer, T_0=10, T_mult=2, eta_min=Config.BASE_LR / 100,
        )

        print(f"\nOptimizer: AdamW (lr={Config.BASE_LR:.2e})")

    def train_epoch(self) -> Tuple[float, float]:
        self.model.train()
        total_loss, total_dice, num_batches = 0.0, 0.0, 0

        pbar = tqdm(self.train_loader, desc="Training", leave=False)
        for batch in pbar:
            try:
                self.optimizer.zero_grad()
                batch = {k: v.to(self.device) for k, v in batch.items()}

                outputs = self.model(**batch)
                loss = outputs.loss

                if torch.isnan(loss) or torch.isinf(loss):
                    continue

                loss.backward()
                torch.nn.utils.clip_grad_norm_(self.model.parameters(), Config.GRAD_CLIP)
                self.optimizer.step()

                with torch.no_grad():
                    if hasattr(outputs, "masks_queries_logits"):
                        # Semantic aggregation over ALL queries weighted by their class
                        # probabilities (query 0 alone is NOT the plume mask; see
                        # aggregate_plume_probability). Averaged over the whole batch.
                        probs = aggregate_plume_probability(outputs)      # [B, h, w]
                        targets = batch["mask_labels"][:, 0]              # [B, h, w]
                        dice = float(sum(
                            Metrics.dice_coefficient(probs[b], targets[b])
                            for b in range(probs.shape[0])
                        ) / probs.shape[0])
                    else:
                        dice = 0.0

                total_loss += loss.item()
                total_dice += dice
                num_batches += 1
                pbar.set_postfix({"Loss": f"{loss.item():.4f}", "Dice": f"{dice:.4f}"})

            except Exception as e:
                print(f"Error: {e}")
                continue

        if num_batches == 0:
            return float("inf"), 0.0
        return total_loss / num_batches, total_dice / num_batches

    @torch.no_grad()
    def validate_epoch(self) -> Tuple[float, float]:
        self.model.eval()
        total_loss, total_dice, num_batches = 0.0, 0.0, 0

        pbar = tqdm(self.val_loader, desc="Validation", leave=False)
        for batch in pbar:
            try:
                batch = {k: v.to(self.device) for k, v in batch.items()}
                outputs = self.model(**batch)
                loss = outputs.loss

                if torch.isnan(loss) or torch.isinf(loss):
                    continue

                if hasattr(outputs, "masks_queries_logits"):
                    # Semantic aggregation over ALL queries weighted by their class
                    # probabilities (query 0 alone is NOT the plume mask; see
                    # aggregate_plume_probability). Averaged over the whole batch.
                    probs = aggregate_plume_probability(outputs)      # [B, h, w]
                    targets = batch["mask_labels"][:, 0]              # [B, h, w]
                    dice = float(sum(
                        Metrics.dice_coefficient(probs[b], targets[b])
                        for b in range(probs.shape[0])
                    ) / probs.shape[0])
                else:
                    dice = 0.0

                total_loss += loss.item()
                total_dice += dice
                num_batches += 1
                pbar.set_postfix({"Loss": f"{loss.item():.4f}", "Dice": f"{dice:.4f}"})

            except Exception:
                continue

        if num_batches == 0:
            return float("inf"), 0.0
        return total_loss / num_batches, total_dice / num_batches

    def save_checkpoint(self, epoch: int, is_best: bool = False):
        try:
            checkpoint = {
                "epoch": epoch + 1,
                "model_state_dict": self.model.state_dict(),
                "optimizer_state_dict": self.optimizer.state_dict(),
                "history": self.history,
            }

            if is_best:
                torch.save(checkpoint, Config.SAVE_PATH)
                print(f"Saved: {Config.SAVE_PATH}")

            if Config.SAVE_CHECKPOINTS and (epoch + 1) % 10 == 0:
                path = f"{Config.PROJECT_DIR}/checkpoint_epoch_{epoch + 1}.pth"
                torch.save(checkpoint, path)

        except Exception as e:
            print(f"Save error: {e}")

    def train(self):
        print("\n" + "=" * 60)
        print("TRAINING START")
        print("=" * 60)

        best_dice = 0.0
        no_improvement = 0

        for epoch in range(Config.EPOCHS):
            epoch_start = time.time()

            train_loss, train_dice = self.train_epoch()
            if train_loss == float("inf"):
                print(f"No valid data in epoch {epoch + 1}")
                break

            val_loss, val_dice = self.validate_epoch()
            if val_loss == float("inf"):
                val_loss, val_dice = train_loss, train_dice

            if self.scheduler:
                self.scheduler.step()

            current_lr = self.optimizer.param_groups[0]["lr"]

            self.history["train_loss"].append(train_loss)
            self.history["val_loss"].append(val_loss)
            self.history["train_dice"].append(train_dice)
            self.history["val_dice"].append(val_dice)
            self.history["learning_rates"].append(current_lr)

            epoch_time = time.time() - epoch_start

            print(f"\n[Epoch {epoch + 1}/{Config.EPOCHS}]")
            print(f"Train - Loss: {train_loss:.4f} | Dice: {train_dice:.4f}")
            print(f"Val   - Loss: {val_loss:.4f} | Dice: {val_dice:.4f}")
            print(f"LR: {current_lr:.2e} | Time: {epoch_time:.1f}s")

            is_best = val_dice > best_dice
            if is_best:
                best_dice = val_dice
                no_improvement = 0
                print(f"New best Dice: {best_dice:.4f}")
            else:
                no_improvement += 1

            self.save_checkpoint(epoch, is_best)

            if no_improvement >= Config.PATIENCE:
                print("\nEarly stopping")
                break

            if (epoch + 1) % 5 == 0:
                Utils.clear_memory()

        if Config.SAVE_PLOTS and len(self.history["train_loss"]) > 0:
            Utils.save_plots(self.history, f"{Config.PROJECT_DIR}/training_plots.png")

        print(f"\n{'=' * 60}")
        print("TRAINING COMPLETED")
        print(f"Best validation Dice: {best_dice:.4f} "
              f"(enter this value in 13_ensemble.ipynb)")
        print("=" * 60)

        return self.model, self.history

    def run(self):
        self.setup_data()
        self.setup_model()
        self.setup_optimizer()
        return self.train()

## 5. Run training

In [ ]:
trainer = Mask2FormerTrainer()
model, history = trainer.run()
Utils.clear_memory()

## 6. Test-set evaluation (canonical metrics)

The best checkpoint is reloaded, the probability mask is taken as the sigmoid
of the first query mask logits, and the same metric set as for the other
models is computed at the native ground-truth resolution.

In [ ]:
import os
from PIL import Image
from scipy.ndimage import distance_transform_edt


def compute_dice(mask_true, mask_pred, eps=1e-7):
    """Dice coefficient between two binary masks (numpy)."""
    mask_true = mask_true.astype(np.float32)
    mask_pred = mask_pred.astype(np.float32)

    intersection = np.sum(mask_true * mask_pred)
    total = np.sum(mask_true) + np.sum(mask_pred)

    if total == 0:
        # Both masks empty: perfect agreement.
        return 1.0

    return (2.0 * intersection) / (total + eps)


def compute_metrics(mask_true, mask_pred, eps=1e-7):
    """Pixel-wise Accuracy, Precision and Recall for binary segmentation."""
    y_true = mask_true.flatten().astype(np.uint8)
    y_pred = mask_pred.flatten().astype(np.uint8)

    TP = np.sum((y_true == 1) & (y_pred == 1))
    FP = np.sum((y_true == 0) & (y_pred == 1))
    FN = np.sum((y_true == 1) & (y_pred == 0))
    TN = np.sum((y_true == 0) & (y_pred == 0))

    total = TP + TN + FP + FN
    accuracy = (TP + TN) / (total + eps)
    precision = 0.0 if (TP + FP) == 0 else TP / (TP + FP)
    recall = 0.0 if (TP + FN) == 0 else TP / (TP + FN)

    return accuracy, precision, recall


def compute_hausdorff_distances(mask_true, mask_pred):
    """HD95, Average HD and Max HD between two binary masks.

    Distances are computed symmetrically with Euclidean distance transforms
    over the full masks of an image.
    Returns (hd95, avg_hd, max_hd) in pixels.
    """
    true_coords = np.argwhere(mask_true > 0)
    pred_coords = np.argwhere(mask_pred > 0)

    if len(true_coords) == 0 or len(pred_coords) == 0:
        if len(true_coords) == 0 and len(pred_coords) == 0:
            return 0.0, 0.0, 0.0
        # One mask is empty: report the image diagonal as the distance.
        max_dist = np.sqrt(mask_true.shape[0] ** 2 + mask_true.shape[1] ** 2)
        return max_dist, max_dist, max_dist

    distances_pred_to_true = distance_transform_edt(1 - mask_true)
    distances_pred_to_true = distances_pred_to_true[pred_coords[:, 0], pred_coords[:, 1]]

    distances_true_to_pred = distance_transform_edt(1 - mask_pred)
    distances_true_to_pred = distances_true_to_pred[true_coords[:, 0], true_coords[:, 1]]

    all_distances = np.concatenate([distances_pred_to_true, distances_true_to_pred])

    hd95 = np.percentile(all_distances, 95)
    avg_hd = np.mean(all_distances)
    max_hd = np.max(all_distances)

    return hd95, avg_hd, max_hd


def binarize_mask(mask, threshold=0.5):
    """Binarize a mask with the given threshold."""
    return (mask > threshold).astype(np.uint8)


def find_corresponding_mask(img_name, mask_files):
    """Find the ground-truth mask file matching an image file name."""
    img_stem = os.path.splitext(img_name)[0]

    possible_names = [
        img_name,
        f"{img_stem}.png",
        f"{img_stem}.jpg",
        f"{img_stem}.jpeg",
        f"{img_stem}_mask.png",
        f"mask_{img_stem}.png",
        f"{img_stem}_gt.png",
        f"{img_stem}_groundtruth.png",
    ]

    for possible_name in possible_names:
        if possible_name in mask_files:
            return possible_name
    return None


def resize_to_match(mask, target_size):
    """Resize a binary mask to target_size with nearest-neighbour resampling."""
    if isinstance(mask, np.ndarray):
        mask_pil = Image.fromarray((mask * 255).astype(np.uint8))
    else:
        mask_pil = mask

    resized = mask_pil.resize(target_size, resample=Image.NEAREST)
    return (np.array(resized) > 127).astype(np.uint8)

In [ ]:
# Load the best checkpoint.
eval_model = Mask2FormerForUniversalSegmentation.from_pretrained(
    Config.PRETRAINED, num_labels=2, ignore_mismatched_sizes=True,
)
checkpoint = torch.load(Config.SAVE_PATH, map_location="cpu")
eval_model.load_state_dict(checkpoint["model_state_dict"])
device = Utils.get_device()
eval_model.to(device)
eval_model.eval()

processor = Mask2FormerImageProcessor.from_pretrained(
    Config.PRETRAINED, do_resize=False, do_normalize=True,
)


@torch.no_grad()
def predict_mask(model, processor, image, device, img_size=Config.IMG_SIZE):
    """Predict a probability mask in [0, 1] for one RGB image (numpy)."""
    image_resized = cv2.resize(image, (img_size, img_size),
                               interpolation=cv2.INTER_LINEAR)
    image_pil = Image.fromarray(image_resized)

    inputs = processor(images=image_pil, return_tensors="pt")
    pixel_values = inputs["pixel_values"].to(device)
    pixel_mask = inputs["pixel_mask"].to(device)

    outputs = model(pixel_values=pixel_values, pixel_mask=pixel_mask)

    # Aggregate ALL queries with their class probabilities; query 0 alone
    # is not the plume mask (see aggregate_plume_probability above).
    prob_mask = aggregate_plume_probability(outputs)[0].cpu().numpy()

    if prob_mask.shape != (img_size, img_size):
        prob_tensor = torch.from_numpy(prob_mask).unsqueeze(0).unsqueeze(0)
        prob_tensor = F.interpolate(prob_tensor, size=(img_size, img_size),
                                    mode="bilinear", align_corners=False)
        prob_mask = prob_tensor.squeeze().numpy()

    return prob_mask


img_names = sorted([f for f in os.listdir(Config.TEST_IMG_DIR)
                    if f.lower().endswith((".png", ".jpg", ".jpeg"))])
mask_files = sorted([f for f in os.listdir(Config.TEST_MASK_DIR)
                     if f.lower().endswith((".png", ".jpg", ".jpeg"))])

dice_scores, accuracy_scores = [], []
precision_scores, recall_scores = [], []
hd95_scores, avg_hd_scores, max_hd_scores = [], [], []

for i, img_name in enumerate(img_names):
    mask_name = find_corresponding_mask(img_name, mask_files)
    if mask_name is None:
        print(f"Mask not found for {img_name}, skipping")
        continue

    bgr = cv2.imread(os.path.join(Config.TEST_IMG_DIR, img_name))
    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    true_mask_img = Image.open(os.path.join(Config.TEST_MASK_DIR, mask_name)).convert("L")
    true_mask_bin = (np.array(true_mask_img) > 127).astype(np.uint8)

    prob = predict_mask(eval_model, processor, image, device)
    pred_bin = binarize_mask(prob, threshold=Config.PRED_THRESHOLD)
    pred_bin = resize_to_match(pred_bin, true_mask_img.size)

    if true_mask_bin.shape != pred_bin.shape:
        min_h = min(true_mask_bin.shape[0], pred_bin.shape[0])
        min_w = min(true_mask_bin.shape[1], pred_bin.shape[1])
        true_mask_bin = true_mask_bin[:min_h, :min_w]
        pred_bin = pred_bin[:min_h, :min_w]

    dice_scores.append(compute_dice(true_mask_bin, pred_bin))
    acc, prec, rec = compute_metrics(true_mask_bin, pred_bin)
    accuracy_scores.append(acc)
    precision_scores.append(prec)
    recall_scores.append(rec)
    hd95, avg_hd, max_hd = compute_hausdorff_distances(true_mask_bin, pred_bin)
    hd95_scores.append(hd95)
    avg_hd_scores.append(avg_hd)
    max_hd_scores.append(max_hd)

    if (i + 1) % 10 == 0:
        print(f"Processed {i + 1}/{len(img_names)}")

print("\n" + "=" * 70)
print("TEST-SET RESULTS: Mask2Former")
print("=" * 70)
print(f"Mean Dice Coefficient: {np.mean(dice_scores):.4f} +/- {np.std(dice_scores):.4f}")
print(f"Mean Accuracy:         {np.mean(accuracy_scores):.4f} +/- {np.std(accuracy_scores):.4f}")
print(f"Mean Precision:        {np.mean(precision_scores):.4f} +/- {np.std(precision_scores):.4f}")
print(f"Mean Recall:           {np.mean(recall_scores):.4f} +/- {np.std(recall_scores):.4f}")
print(f"Mean HD95:             {np.mean(hd95_scores):.4f} +/- {np.std(hd95_scores):.4f}")
print(f"Mean Average HD:       {np.mean(avg_hd_scores):.4f} +/- {np.std(avg_hd_scores):.4f}")
print(f"Mean Max HD:           {np.mean(max_hd_scores):.4f} +/- {np.std(max_hd_scores):.4f}")

## 7. Export predictions for the ensemble

In [ ]:
os.makedirs(Config.PROBABILITY_EXPORT_DIR, exist_ok=True)
os.makedirs(Config.BINARY_EXPORT_DIR, exist_ok=True)

for img_name in img_names:
    img_stem = os.path.splitext(img_name)[0]

    bgr = cv2.imread(os.path.join(Config.TEST_IMG_DIR, img_name))
    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    prob = predict_mask(eval_model, processor, image, device)

    prob_png = (prob * 255).astype(np.uint8)
    cv2.imwrite(os.path.join(Config.PROBABILITY_EXPORT_DIR, f"{img_stem}_pred.png"), prob_png)

    binary_png = ((prob > Config.PRED_THRESHOLD).astype(np.uint8) * 255)
    cv2.imwrite(os.path.join(Config.BINARY_EXPORT_DIR, f"{img_stem}_pred.png"), binary_png)

print(f"Probability masks written to: {Config.PROBABILITY_EXPORT_DIR}")
print(f"Binary masks written to:      {Config.BINARY_EXPORT_DIR}")

## 8. Export validation predictions

Same convention as the test export. Used by `13_ensemble.ipynb` (ensemble
weights and scheme selection on the validation split),
`14_stacking_random_forest.ipynb` and the uniform validation-Dice table.

In [ ]:
VAL_IMG_DIR = "split_data/val/images"
VAL_PROBABILITY_EXPORT_DIR = "probability_masks/val/mask2former"
VAL_BINARY_EXPORT_DIR = "val_masks/mask2former"

os.makedirs(VAL_PROBABILITY_EXPORT_DIR, exist_ok=True)
os.makedirs(VAL_BINARY_EXPORT_DIR, exist_ok=True)

val_img_names = sorted([f for f in os.listdir(VAL_IMG_DIR)
                        if f.lower().endswith((".png", ".jpg", ".jpeg"))])

for img_name in val_img_names:
    img_stem = os.path.splitext(img_name)[0]

    bgr = cv2.imread(os.path.join(VAL_IMG_DIR, img_name))
    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    prob = predict_mask(eval_model, processor, image, device)

    cv2.imwrite(os.path.join(VAL_PROBABILITY_EXPORT_DIR, f"{img_stem}_pred.png"),
                (prob * 255).astype(np.uint8))
    cv2.imwrite(os.path.join(VAL_BINARY_EXPORT_DIR, f"{img_stem}_pred.png"),
                ((prob > Config.PRED_THRESHOLD).astype(np.uint8) * 255))

print(f"Validation probability masks written to: {VAL_PROBABILITY_EXPORT_DIR}")
print(f"Validation binary masks written to:      {VAL_BINARY_EXPORT_DIR}")

## 9. Test-time-augmented export (optional)

Model-level TTA over six dihedral transforms of the input; predictions are
mapped back to the original orientation and averaged. Consumed by the TTA
protocol of `15_postprocessing_methods.ipynb`.

In [ ]:
TTA_MODES = ("identity", "hflip", "vflip", "rot90", "rot180", "rot270")

def tta_apply(a, mode):
    if mode == "hflip":
        return a[:, ::-1]
    if mode == "vflip":
        return a[::-1, :]
    if mode == "rot90":
        return np.rot90(a, 1)
    if mode == "rot180":
        return np.rot90(a, 2)
    if mode == "rot270":
        return np.rot90(a, 3)
    return a

def tta_invert(a, mode):
    if mode == "hflip":
        return a[:, ::-1]
    if mode == "vflip":
        return a[::-1, :]
    if mode == "rot90":
        return np.rot90(a, -1)
    if mode == "rot180":
        return np.rot90(a, -2)
    if mode == "rot270":
        return np.rot90(a, -3)
    return a

TTA_PROBABILITY_EXPORT_DIR = "probability_masks/test_tta/mask2former"
os.makedirs(TTA_PROBABILITY_EXPORT_DIR, exist_ok=True)

for img_name in img_names:
    img_stem = os.path.splitext(img_name)[0]

    bgr = cv2.imread(os.path.join(Config.TEST_IMG_DIR, img_name))
    image = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)

    acc = None
    for mode in TTA_MODES:
        t = np.ascontiguousarray(tta_apply(image, mode))
        p = predict_mask(eval_model, processor, t, device)
        p = np.ascontiguousarray(tta_invert(p, mode)).astype(np.float64)
        acc = p if acc is None else acc + p

    prob = acc / len(TTA_MODES)
    cv2.imwrite(os.path.join(TTA_PROBABILITY_EXPORT_DIR, f"{img_stem}_pred.png"),
                (prob * 255).astype(np.uint8))

print(f"TTA probability masks written to: {TTA_PROBABILITY_EXPORT_DIR}")